# 29 — Hardware-Aware Scheduling

**Engineering statement:** Hardware constraints specify feasible verification schedules.

Notebook 23 defined the throughput objective. Notebook 29 asks what changes when the objective is executed on real serving hardware: active batch size, KV-cache pressure, memory bandwidth, verification latency, and queue pressure restrict which schedules remain feasible.


## Repository roadmap

```text
00 Context
07 Verification Resource
13 Confidence Scheduling
17 Semi-Autoregressive Design
23 Throughput Objective
29 Hardware-Aware Scheduling
37 Operating Regimes
43 Resource Allocation
49 Adaptive AI Infrastructure
```


## Start here

```text
throughput objective
→ active batch size
→ memory pressure
→ verification latency
→ feasible schedule set
→ hardware-aware policy
→ serving throughput
```

Notebook 23 optimized an abstract objective,

\[
\Theta(\ell)=A(\ell)S(B).
\]

Notebook 29 adds the constraint that \(S(B)\) is not a smooth free variable. It depends on concrete serving limits: batch size, memory footprint, cache traffic, and per-step latency.


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from IPython.display import display, FileLink

# Paths compatible with repo root, notebooks/, or standalone execution.
CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "29_hardware_aware_scheduling"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=14):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.5,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle="->", lw=1.8))


## 1. Hardware constraints enter the scheduler

Confidence scheduling selects useful verification prefixes. Hardware-aware scheduling asks whether those prefixes are feasible under the current serving state.

The scheduler no longer sees only

```text
prefix survival → expected accepted tokens
```

It sees

```text
prefix survival + active batch + memory pressure + latency → feasible schedule
```


In [ ]:
# 29_hardware_constraint_flow.png

fig, ax = plt.subplots(figsize=(13, 4.2))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 4)

nodes = [
    ("Throughput\nobjective", 0.4, 2.4, 1.7, 0.8),
    ("Active\nbatch", 2.8, 2.4, 1.4, 0.8),
    ("Memory\npressure", 4.9, 2.4, 1.5, 0.8),
    ("Verification\nlatency", 7.0, 2.4, 1.6, 0.8),
    ("Feasible\nschedule set", 9.2, 2.4, 1.8, 0.8),
    ("Hardware-aware\npolicy", 11.4, 2.4, 1.6, 0.8),
]
for label, x, y, w, h in nodes:
    rounded_box(ax, (x, y), w, h, label, fontsize=13)

for i in range(len(nodes)-1):
    _, x, y, w, h = nodes[i]
    _, x2, y2, w2, h2 = nodes[i+1]
    arrow(ax, (x+w+0.08, y+h/2), (x2-0.08, y2+h2/2))

rounded_box(ax, (4.2, 0.55), 4.6, 0.7, "Hardware turns the objective into a constrained scheduling problem.", fontsize=12)
arrow(ax, (6.5, 2.35), (6.5, 1.3))

ax.set_title("Hardware constraints specify feasible verification schedules", fontsize=22, pad=18)
savefig("29_hardware_constraint_flow.png")


## 2. Batch size changes verification latency

A scheduled prefix length \(\ell\) is attractive only if it remains efficient at the current active batch size. At small \(R\), longer verification can increase accepted tokens. At larger \(R\), the same prefix length may increase batch pressure and reduce steps per second.

The feasible schedule is therefore load-dependent.


In [ ]:
# 29_batch_latency_curve.png

R = np.arange(1, 65)
ells = np.array([1, 2, 4, 6, 8])

def latency(R, ell):
    base = 1.0
    batch_term = 0.015 * R**1.12
    verify_term = 0.055 * ell
    interaction = 0.0025 * R * ell
    saturation = 0.00005 * (R * ell) ** 1.35
    return base + batch_term + verify_term + interaction + saturation

fig, ax = plt.subplots(figsize=(10, 5.6))
for ell in ells:
    ax.plot(R, latency(R, ell), marker="o", markevery=6, label=fr"$\ell={ell}$")
ax.set_title("Verification latency increases with active batch and scheduled prefix length", fontsize=20)
ax.set_xlabel("active requests R", fontsize=14)
ax.set_ylabel("latency proxy", fontsize=14)
ax.legend(ncol=3)
ax.grid(True, alpha=0.35)
savefig("29_batch_latency_curve.png")

# Save data
data = {"R": R.tolist(), **{f"latency_l{ell}": latency(R, ell).round(4).tolist() for ell in ells}}
(RESULTS / "29_batch_latency_curve.json").write_text(json.dumps(data, indent=2), encoding="utf-8")


## 3. Memory pressure restricts schedule length

Verification is not only compute. It also carries memory pressure: active requests, scheduled verification tokens, and KV/cache traffic all interact.

A schedule can be statistically attractive and still be operationally infeasible.

\[
M(R,\ell)\le M_{\max}
\]


In [ ]:
# 29_memory_pressure.png

R_vals = np.arange(4, 65, 4)
ell_vals = np.arange(1, 9)
Mmax = 1.0

def memory_pressure(R, ell):
    return 0.18 + 0.0065 * R + 0.045 * ell + 0.0018 * R * ell

grid = np.array([[memory_pressure(R, ell) for ell in ell_vals] for R in R_vals])

fig, ax = plt.subplots(figsize=(10, 5.8))
im = ax.imshow(grid, aspect="auto", origin="lower", extent=[ell_vals.min(), ell_vals.max(), R_vals.min(), R_vals.max()])
ax.contour(ell_vals, R_vals, grid, levels=[Mmax], linewidths=2)
ax.set_title("Memory pressure limits feasible verification schedules", fontsize=20)
ax.set_xlabel("scheduled verification length $\\ell$", fontsize=14)
ax.set_ylabel("active requests R", fontsize=14)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("memory pressure proxy")
ax.text(6.3, 50, "infeasible\nregion", fontsize=12)
ax.text(1.5, 16, "feasible\nregion", fontsize=12)
savefig("29_memory_pressure.png")


## 4. Feasibility map

Notebook 23 selected \(\ell^*\) from throughput alone. Notebook 29 selects from the feasible set,

\[
\mathcal{F}(R,M_{\max},L_{\max})
=
\{\ell:\;M(R,\ell)\le M_{\max},\;L(R,\ell)\le L_{\max}\}.
\]

The schedule is now

\[
\ell^*
=
\arg\max_{\ell\in\mathcal{F}}
\Theta(\ell).
\]


In [ ]:
# 29_schedule_feasibility_map.png

R_grid = np.arange(4, 65, 2)
q_grid = np.linspace(0.2, 0.9, 60)  # draft predictability / confidence floor proxy
E = np.arange(1, 9)

def accepted(q, ell):
    c = np.clip(q + 0.18*np.exp(-0.15*np.arange(ell)), 0.05, 0.98)
    return np.cumprod(c).sum()

def steps_per_sec(R, ell):
    return 1000 / latency(R, ell)

selected = np.zeros((len(R_grid), len(q_grid)))
feasible_count = np.zeros_like(selected)

for i, R0 in enumerate(R_grid):
    for j, q in enumerate(q_grid):
        vals = []
        feasible = []
        for ell in E:
            ok = (memory_pressure(R0, ell) <= 1.0) and (latency(R0, ell) <= 3.3)
            feasible.append(ok)
            vals.append(accepted(q, ell) * steps_per_sec(R0, ell) if ok else -np.inf)
        selected[i, j] = E[int(np.argmax(vals))]
        feasible_count[i, j] = sum(feasible)

fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(
    selected, aspect="auto", origin="lower",
    extent=[q_grid.min(), q_grid.max(), R_grid.min(), R_grid.max()]
)
ax.set_title("Feasible hardware-aware schedule length", fontsize=20)
ax.set_xlabel("draft predictability / confidence floor", fontsize=14)
ax.set_ylabel("active requests R", fontsize=14)
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("selected $\\ell$")
savefig("29_schedule_feasibility_map.png")


## 5. Hardware-aware policy

The policy now has three inputs:

1. **Prefix survival:** what verification is expected to accept.
2. **Serving pressure:** how many active requests share target-model capacity.
3. **Hardware feasibility:** whether memory and latency constraints admit a schedule.

A hardware-aware scheduler shortens prefixes under load and lengthens them when feasible.


In [ ]:
# 29_hardware_aware_policy.png

request_types = ["chat", "instruction", "code", "math"]
q = np.array([0.36, 0.46, 0.58, 0.64])
loads = [8, 24, 48]
E = np.arange(1, 9)

def choose_ell(qval, R0):
    vals = []
    for ell in E:
        ok = (memory_pressure(R0, ell) <= 1.0) and (latency(R0, ell) <= 3.3)
        vals.append(accepted(qval, ell) * steps_per_sec(R0, ell) if ok else -np.inf)
    return int(E[np.argmax(vals)])

choices = np.array([[choose_ell(qv, R0) for qv in q] for R0 in loads])

x = np.arange(len(request_types))
width = 0.24
fig, ax = plt.subplots(figsize=(10, 5.8))
for i, R0 in enumerate(loads):
    ax.bar(x + (i-1)*width, choices[i], width=width, label=f"R={R0}")
ax.set_xticks(x, request_types)
ax.set_ylabel("selected scheduled length $\\ell^*$", fontsize=14)
ax.set_title("Hardware-aware policy shortens schedules under load", fontsize=20)
ax.legend(title="active requests")
ax.grid(axis="y", alpha=0.3)
savefig("29_hardware_aware_policy.png")

policy_data = {
    "request_types": request_types,
    "loads": loads,
    "selected_lengths": choices.tolist()
}
(RESULTS / "29_hardware_aware_policy.json").write_text(json.dumps(policy_data, indent=2), encoding="utf-8")


## 6. Serving controller architecture

A deployable confidence-scheduled system needs a controller. The controller observes batch state, memory pressure, and confidence distributions; then it chooses verification schedules before target-model verification.


In [ ]:
# 29_serving_controller.png

fig, ax = plt.subplots(figsize=(13, 5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5)

top = [
    ("Requests", 0.4, 3.3, 1.3, 0.75),
    ("Drafter", 2.3, 3.3, 1.3, 0.75),
    ("Confidence\nprofiles", 4.2, 3.3, 1.5, 0.75),
    ("Schedule\ncontroller", 6.3, 3.1, 1.8, 1.1),
    ("Target\nverification", 8.9, 3.3, 1.6, 0.75),
    ("Accepted\ntokens", 11.2, 3.3, 1.4, 0.75),
]
for label, x0, y0, w, h in top:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=12)

for i in range(len(top)-1):
    _, x0, y0, w, h = top[i]
    _, x1, y1, w1, h1 = top[i+1]
    arrow(ax, (x0+w+0.05, y0+h/2), (x1-0.05, y1+h1/2))

signals = [
    ("active batch R", 4.8, 1.15, 1.6, 0.65),
    ("memory pressure", 6.6, 1.15, 1.8, 0.65),
    ("latency target", 8.7, 1.15, 1.5, 0.65),
]
for label, x0, y0, w, h in signals:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=11)
    arrow(ax, (x0+w/2, y0+h+0.05), (7.2, 3.05))

ax.set_title("Hardware-aware confidence-scheduled serving controller", fontsize=22, pad=12)
savefig("29_serving_controller.png")


## 7. Transition to operating regimes

Notebook 29 adds constraints. Notebook 37 can now study operating regimes: low-load, memory-limited, latency-limited, throughput-limited, and adaptive serving regimes.

The transition is:

```text
objective
→ feasible schedules
→ hardware-aware policy
→ operating regimes
```


In [ ]:
# 29_transition_operating_regimes.png

fig, ax = plt.subplots(figsize=(12, 3.8))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)

nodes = [
    ("Throughput\nobjective", 0.6, 1.35, 1.6, 0.7),
    ("Feasible\nschedules", 3.0, 1.35, 1.6, 0.7),
    ("Hardware-aware\npolicy", 5.4, 1.35, 1.8, 0.7),
    ("Operating\nregimes", 8.1, 1.35, 1.5, 0.7),
    ("Adaptive\nserving", 10.3, 1.35, 1.4, 0.7),
]
for label, x0, y0, w, h in nodes:
    rounded_box(ax, (x0, y0), w, h, label, fontsize=12)

for i in range(len(nodes)-1):
    _, x0, y0, w, h = nodes[i]
    _, x1, y1, w1, h1 = nodes[i+1]
    arrow(ax, (x0+w+0.06, y0+h/2), (x1-0.06, y1+h1/2))

ax.set_title("Notebook transition: constraints become operating regimes", fontsize=20, pad=12)
savefig("29_transition_operating_regimes.png")


## Key equations

Prefix survival:

\[
a_j=\prod_{k=1}^{j}c_k
\]

Expected accepted tokens:

\[
A(\ell)=\sum_{j=1}^{\ell}a_j
\]

Active verification batch:

\[
B=\sum_{r=1}^{R}(1+\ell_r)
\]

Latency constraint:

\[
L(R,\ell)\le L_{\max}
\]

Memory constraint:

\[
M(R,\ell)\le M_{\max}
\]

Feasible schedule set:

\[
\mathcal{F}
=
\{\ell:\;L(R,\ell)\le L_{\max},\;M(R,\ell)\le M_{\max}\}
\]

Hardware-aware schedule:

\[
\ell^*
=
\arg\max_{\ell\in\mathcal{F}}
A(\ell)S(B).
\]


## Engineering summary

Notebook 23 asked which schedule maximizes throughput.

Notebook 29 adds the missing systems constraint:

> The best statistical schedule is not always the best deployable schedule.

Hardware-aware scheduling selects from the schedules that are feasible under active batch size, memory pressure, and latency constraints.


In [ ]:
# Save a notebook-level summary and manifest

summary = {
    "notebook": "29_hardware_aware_scheduling.ipynb",
    "title": "Hardware-Aware Scheduling",
    "engineering_statement": "Hardware constraints specify feasible verification schedules.",
    "figures": [
        "29_hardware_constraint_flow.png",
        "29_batch_latency_curve.png",
        "29_memory_pressure.png",
        "29_schedule_feasibility_map.png",
        "29_hardware_aware_policy.png",
        "29_serving_controller.png",
        "29_transition_operating_regimes.png",
    ],
    "next_notebook": "37_operating_regimes.ipynb",
}
(RESULTS / "29_hardware_aware_scheduling_manifest.json").write_text(
    json.dumps(summary, indent=2), encoding="utf-8"
)
summary


## Download artifacts

Run the final cell to package Notebook 29 outputs for download.

The archive includes:

- `29_hardware_aware_scheduling.ipynb`
- generated figures
- generated JSON outputs
- notebook manifest


In [ ]:
# Package Notebook 29 artifacts for download

from pathlib import Path
from IPython.display import FileLink, display
import zipfile
import json

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "29_hardware_aware_scheduling"
SITE = REPO_ROOT / "site"

RESULTS.mkdir(parents=True, exist_ok=True)

zip_path = RESULTS / "29_hardware_aware_scheduling_artifacts.zip"

notebook_candidates = [
    NOTEBOOKS / "29_hardware_aware_scheduling.ipynb",
    Path.cwd() / "29_hardware_aware_scheduling.ipynb",
]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)

for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file():
                    continue
                if path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
